# Baseline on the prepared clinical dataset

Run **prepare_clinical_dataset.ipynb first**. This evaluates the unmodified Gemma checkpoint on the exact saved validation cases and prompt texts. It never resamples, repairs generated JSON, trains, or sends data to a judge. It preserves raw generation diagnostics and computes local metrics that exclude missing-value placeholders and schema words.

Defaults: all 10 prepared validation cases, greedy decoding, 2,048 output tokens, FP32 (safe diagnostic default including T4). GPU is useful for generation. Set INFERENCE_DTYPE to `bfloat16` only on hardware with native BF16 support. Keep the generated comparison contract and artifacts for the fine-tuned evaluation; do not compare these numbers directly to the older schema-prompt experiments.

Structural JSON checks and exact normalized value matching are separate. Value matching ignores array order but counts duplicate values, and is strict about wording: it is not a clinical or semantic accuracy score. Invalid JSON gets zero extracted-value matches. Syntactically valid placeholder-only output is explicitly flagged. No paid judging is configured.

## Dataset location when using VS Code with Colab

The Python kernel reads files on the **remote Colab machine**. A folder visible in the local VS Code Explorer is not automatically available there. If you changed runtimes after preparation, upload the saved `prepared_clinical_dataset.zip` to the remote runtime first. Do not prepare another dataset merely to replace missing files.

After uploading the archive to `/content`, run this in a temporary cell before setup:

```python
import shutil
from pathlib import Path
archive = Path("/content/prepared_clinical_dataset.zip")
assert archive.is_file(), "Upload the saved dataset ZIP to the remote runtime first."
shutil.unpack_archive(str(archive), "/content", "zip")
assert Path("/content/prepared_clinical_dataset/latest.json").is_file()
```

The ZIP created by the preparation notebook includes the top-level folder. Set `PREPARED_ROOT = Path("/content/prepared_clinical_dataset")` if using a different working directory, or configure `PREPARED_CLINICAL_ROOT` in the kernel environment. The setup cell prints the working directory and resolved dataset root and checks that the versioned dataset was extracted completely.


In [3]:
%pip install --progress-bar on "transformers==4.57.6" "datasets>=3,<5" huggingface_hub pandas accelerate


In [4]:
import os, json, hashlib, random, re, time, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
from huggingface_hub import HfApi, get_token
from transformers import AutoTokenizer

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        print("Enter your Hugging Face token in the VS Code input prompt.", flush=True)
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM
# This path belongs to the machine running the kernel, not necessarily your PC.
PREPARED_ROOT = Path(os.environ.get("PREPARED_CLINICAL_ROOT", "prepared_clinical_dataset"))
DATASET_VERSION = None  # Set a specific version string to override latest.json.
INFERENCE_DTYPE = "float32"
MAX_NEW_TOKENS = 2048
OUTPUT_ROOT = Path("baseline_prepared_evaluation")
SEED = 42
assert MAX_NEW_TOKENS > 0
def resolve_prepared_root(configured):
    configured = Path(configured).expanduser()
    candidates = [configured]
    if configured == Path("prepared_clinical_dataset"):
        candidates += [parent / configured for parent in Path.cwd().parents]
        candidates.append(Path("/content/prepared_clinical_dataset"))
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "latest.json").is_file():
            return candidate.resolve()
    checked = "\n".join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(
        f"Prepared dataset is not visible to this kernel. Working directory: {Path.cwd()}\n"
        f"Checked for latest.json under:\n{checked}\n"
        "VS Code local files are not automatically copied to a remote Colab kernel. "
        "Upload prepared_clinical_dataset.zip to the runtime, extract it there, and set "
        "PREPARED_ROOT to the extracted folder (for example /content/prepared_clinical_dataset). "
        "Do not rerun preparation or select new cases to fix a missing path.")

print("Kernel working directory:", Path.cwd(), flush=True)
PREPARED_ROOT = resolve_prepared_root(PREPARED_ROOT)
print("Resolved prepared dataset root:", PREPARED_ROOT, flush=True)
version = DATASET_VERSION or json.loads((PREPARED_ROOT / "latest.json").read_text(encoding="utf-8"))["version"]
if not isinstance(version, str) or Path(version).name != version or version in (".", ".."):
    raise ValueError("Dataset version must be a single directory name")
prepared_dir = PREPARED_ROOT / version
for required in (prepared_dir / "manifest.json", prepared_dir / "dataset" / "dataset_dict.json"):
    if not required.exists():
        raise FileNotFoundError(f"Prepared dataset extraction is incomplete: missing {required}")
manifest = json.loads((prepared_dir / "manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir / "dataset"))
for split, expected_hash in manifest["split_sha256"].items():
    assert digest(prepared[split].to_list()) == expected_hash, f"Prepared {split} content changed."
cases = prepared["validation"].to_list()
assert cases
print("Evaluation cases:", len(cases), "; dataset version:", version)

token = read_token()
MODEL_ID = manifest["config"]["model"]
model_revision = manifest["config"]["model_revision"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=token)
device = "cuda" if torch.cuda.is_available() else "cpu"
if INFERENCE_DTYPE == "bfloat16":
    if device != "cuda" or not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError("Native BF16 not supported; select float32.")
    dtype = torch.bfloat16
elif INFERENCE_DTYPE == "float32":
    dtype = torch.float32
else:
    raise ValueError("Select float32 or bfloat16; FP16 is intentionally excluded.")
# This contract must be reused for later fine-tuned evaluation.
contract = {"dataset_version": version, "validation_sha256": manifest["split_sha256"]["validation"],
            "model": MODEL_ID, "base_model_revision": model_revision,
            "case_ids": [c["case_id"] for c in cases],
            "prompt_sha256": digest([c["prompt_text"] for c in cases]),
            "tokenizer_add_special_tokens": False, "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False, "seed": SEED, "dtype": INFERENCE_DTYPE,
            "metrics_version": 1, "transformers": importlib.metadata.version("transformers"),
            "torch": torch.__version__}
output_dir = OUTPUT_ROOT / (version + "_" + digest(contract)[:12])
output_dir.mkdir(parents=True, exist_ok=True)
config_path = output_dir / "comparison_contract.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == contract
else:
    config_path.write_text(json.dumps(contract, indent=2), encoding="utf-8")
(output_dir / "selection.json").write_text(json.dumps([
    {k:c[k] for k in ("case_id", "source_index", "source_sha256", "system", "note", "reference", "prompt_text")}
    for c in cases], ensure_ascii=False, indent=2), encoding="utf-8")
print("Loading original model on", device, "with", dtype, flush=True)
random.seed(SEED)
torch.manual_seed(SEED)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=model_revision,
                                            token=token, dtype=dtype).to(device).eval()


Kernel working directory: /content
Resolved prepared dataset root: /content/prepared_clinical_dataset
Evaluation cases: 10 ; dataset version: 900825a06d38a006
Enter your Hugging Face token in the VS Code input prompt.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loading original model on cuda with torch.float32


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
generations_path = output_dir / "generations.json"
diagnostics_path = output_dir / "generation_diagnostics.json"
generations = json.loads(generations_path.read_text()) if generations_path.exists() else {}
diagnostics = json.loads(diagnostics_path.read_text()) if diagnostics_path.exists() else {}
started = time.monotonic()
for case in cases:
    key = case["case_id"]
    if generations.get(key, "").strip() and key in diagnostics:
        continue
    inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
    n_prompt = inputs.input_ids.shape[1]
    if n_prompt + MAX_NEW_TOKENS > model.config.max_position_embeddings:
        raise ValueError(f"{key}: prompt plus generation budget exceeds model context")
    with torch.inference_mode():
        # Keep only the final position to avoid materializing all prompt logits.
        logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
        finite = bool(torch.isfinite(logits).all().item())
        del logits
        if finite:
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
            ids = generated[0, n_prompt:].tolist()
        else:
            ids = []
    text = tokenizer.decode(ids, skip_special_tokens=True)
    eos = model.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    diagnostics[key] = {"prompt_tokens": n_prompt, "generated_tokens": len(ids),
        "finite_first_token_logits": finite, "empty_output": not text.strip(),
        "reached_budget": len(ids) >= MAX_NEW_TOKENS,
        "ended_with_eos": bool(ids) and ids[-1] in eos,
        "token_ids": ids, "raw_decoded": tokenizer.decode(ids, skip_special_tokens=False)}
    diagnostics_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
    if not finite or not text.strip():
        raise RuntimeError(f"{key}: failed generation. Diagnostics saved; rerun retries this case.")
    generations[key] = text
    generations_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{len(generations)}/{len(cases)}: {key}, {len(ids)} tokens, {time.monotonic()-started:.1f}s; {text[:100]!r}", flush=True)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/10: validation:2761, 232 tokens, 15.8s; '```json\n{\n  "visit_motivation": "To provide a comprehensive evaluation of the patient\'s presentation'
2/10: validation:465, 379 tokens, 29.2s; '```json\n{\n  "visit motivation": "To understand the patient\'s medical history, current condition, and'
3/10: validation:105, 395 tokens, 43.0s; '```json\n{\n  "visit motivation": "To manage the patient\'s pain and improve his vision.",\n  "admission'


In [ ]:
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge")
FORMAT_INSTRUCTION = (
    "Extract the clinical facts stated in the note into one JSON object. "
    "Return JSON only, without markdown or commentary. Use these top-level fields: "
    + ", ".join(TOP_LEVEL_KEYS) + ". "
    "Use the string None only for information absent from the note. "
    "Do not invent facts or repeat identical entries.")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(text, strict=strict, object_pairs_hook=no_duplicate_keys,
                     parse_constant=reject_constant)
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def clean_reference(raw):
    text = raw.strip()
    fenced = text.startswith("```") and text.endswith("```")
    if fenced:
        text = re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    try:
        obj = load_object(text)
        repair = "removed_markdown_fence" if fenced else "none"
    except ValueError:
        # Only unescaped control characters are tolerated; serialization escapes them.
        # No guessed commas, missing braces, invented values, or LLM repair.
        obj = load_object(text, strict=False)
        repair = "escaped_control_characters" + ("_and_removed_fence" if fenced else "")
    if set(obj) != set(TOP_LEVEL_KEYS):
        raise ValueError("Unexpected top-level schema")
    canonical = json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    assert load_object(canonical) == obj
    return canonical, repair

def strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        return re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    return text

def facts(value, path=""):
    """Multiset of (field path, normalized value); array order is ignored."""
    result = Counter()
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(facts(child, f"{path}.{key}" if path else key))
    elif isinstance(value, list):
        for child in value:
            result.update(facts(child, path))
    elif value is not None:
        normalized = " ".join(str(value).casefold().split())
        if normalized not in ("", "none", "null", "unknown", "not provided"):
            result[(path, normalized)] += 1
    return result

def key_paths(value, prefix=""):
    result = set()
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key
            result.add(path)
            result.update(key_paths(child, path))
    elif isinstance(value, list):
        for child in value:
            result.update(key_paths(child, prefix))
    return result

def evaluate_output(text, reference):
    expected = load_object(reference)
    raw_valid = True
    try:
        load_object(text)
    except ValueError:
        raw_valid = False
    try:
        actual = load_object(strip_fences(text))
        error = None
    except ValueError as exc:
        actual, error = None, str(exc)
    expected_facts = facts(expected)
    actual_facts = facts(actual)
    matches = sum((expected_facts & actual_facts).values())
    n_actual, n_expected = sum(actual_facts.values()), sum(expected_facts.values())
    precision = matches / n_actual if n_actual else 0.0
    recall = matches / n_expected if n_expected else 0.0
    return {"empty": not text.strip(), "raw_json_object_valid": raw_valid,
            "json_object_valid_after_fence_removal": actual is not None,
            "parse_error": error, "exact_key_match": actual is not None and key_paths(actual) == key_paths(expected),
            "nonmissing_generated_values": n_actual, "reference_values": n_expected,
            "matched_values": matches, "placeholder_only": actual is not None and n_actual == 0,
            "value_precision": precision, "value_recall": recall,
            "value_f1": 2*precision*recall/(precision+recall) if precision+recall else 0.0}

ref = '{"age":"29", "symptoms":[{"name":"pain"}]}'
assert evaluate_output('```json\n'+ref+'\n```', ref)["value_f1"] == 1
assert not evaluate_output('```json\n'+ref+'\n```', ref)["raw_json_object_valid"]
assert evaluate_output('{"age":"None"}', ref)["placeholder_only"]
assert evaluate_output('', ref)["value_recall"] == 0
assert evaluate_output('{"age":"29","age":"30"}', ref)["parse_error"]
assert evaluate_output('{"age":"29", "symptoms":[{"name":"pain"},{"name":"pain"}]}', ref)["value_precision"] < 1
print("Metric self-checks passed.")


Metric self-checks passed.


In [ ]:
rows = []
for case in cases:
    key = case["case_id"]
    if key not in generations:
        continue
    result = evaluate_output(generations[key], case["reference"])
    diag = diagnostics.get(key, {})
    rows.append({"case_id": key, "source_index": case["source_index"], **result,
                 "generated_tokens": diag.get("generated_tokens"), "reached_budget": diag.get("reached_budget"),
                 "generated": generations[key], "reference": case["reference"]})
results = pd.DataFrame(rows)
results.to_csv(output_dir / "per_case.csv", index=False)
summary = {"selected_cases": len(cases), "evaluated_cases": len(rows)}
for name in ("empty", "raw_json_object_valid", "json_object_valid_after_fence_removal", "exact_key_match",
             "placeholder_only", "reached_budget"):
    summary[name + "_cases"] = int(results[name].fillna(False).sum()) if rows else 0
for name in ("value_precision", "value_recall", "value_f1"):
    summary["mean_" + name] = float(results[name].mean()) if rows else None
summary["matched_values"] = int(results.matched_values.sum()) if rows else 0
summary["reference_values"] = int(results.reference_values.sum()) if rows else 0
summary["generated_values"] = int(results.nonmissing_generated_values.sum()) if rows else 0
(output_dir / "aggregate.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
if rows:
    display(results.drop(columns=["generated", "reference"]))
print("Keep comparison_contract.json, selection.json, generations.json, and metrics for the fine-tuned comparison.")


{
  "selected_cases": 10,
  "evaluated_cases": 10,
  "empty_cases": 0,
  "raw_json_object_valid_cases": 0,
  "json_object_valid_after_fence_removal_cases": 4,
  "exact_key_match_cases": 0,
  "placeholder_only_cases": 0,
  "reached_budget_cases": 3,
  "mean_value_precision": 0.006666666666666666,
  "mean_value_recall": 0.002,
  "mean_value_f1": 0.003076923076923077,
  "matched_values": 1,
  "reference_values": 353,
  "generated_values": 57
}


,case_id,source_index,empty,raw_json_object_valid,json_object_valid_after_fence_removal,parse_error,exact_key_match,nonmissing_generated_values,reference_values,matched_values,placeholder_only,value_precision,value_recall,value_f1,generated_tokens,reached_budget
0,validation:2761,2761,False,False,True,None,False,9,32,0,False,0.000000,0.00,0.000000,232,False
1,validation:465,465,False,False,True,None,False,18,23,0,False,0.000000,0.00,0.000000,379,False
2,validation:105,105,False,False,True,None,False,15,34,0,False,0.000000,0.00,0.000000,395,False
3,validation:1161,1161,False,False,False,Expecting ':' delimiter: line 12 column 94 (ch...,False,0,23,0,False,0.000000,0.00,0.000000,1072,False
4,validation:1030,1030,False,False,False,Expecting value: line 1 column 1 (char 0),False,0,23,0,False,0.000000,0.00,0.000000,2048,True
5,validation:935,935,False,False,True,None,False,15,50,1,False,0.066667,0.02,0.030769,320,False
6,validation:583,583,False,False,False,Expecting value: line 1 column 1 (char 0),False,0,40,0,False,0.000000,0.00,0.000000,2048,True
7,validation:428,428,False,False,False,Illegal trailing comma before end of object: l...,False,0,47,0,False,0.000000,0.00,0.000000,706,False
8,validation:2926,2926,False,False,False,Duplicate JSON key: surgery_details,False,0,58,0,False,0.000000,0.00,0.000000,1243,False
9,validation:2336,2336,False,False,False,Expecting value: line 1 column 1 (char 0),False,0,23,0,False,0.000000,0.00,0.000000,2048,True


Keep comparison_contract.json, selection.json, generations.json, and metrics for the fine-tuned comparison.


In [ ]:
import shutil
archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir.parent,
                             base_dir=output_dir.name)
print("Download baseline artifacts:", archive)
# Browser Colab only: from google.colab import files; files.download(archive)


Download baseline artifacts: /content/baseline_prepared_evaluation/900825a06d38a006_353c8393d011.zip
